# Return Sign Prediction - Complete Pipeline

**Challenge**: Binary classification of next-day return sign

**Objective**: Predict whether an allocation's return will be positive (1) or negative (0)

**Data**: 527k training observations, 31k test observations

**Metric**: Accuracy

---

## 1. Setup and Configuration

In [ ]:
# Imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configuration
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../models')
SEED = 42
np.random.seed(SEED)

# Feature columns
RET_COLS = [f'RET_{i}' for i in range(1, 21)]
VOL_COLS = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
STATIC_COLS = ['MEDIAN_DAILY_TURNOVER', 'GROUP']

print('✓ Setup complete')

## 2. Load Data

In [ ]:
# Load datasets
X_train_raw = pd.read_csv(DATA_DIR / 'X_train.csv', index_col='ROW_ID')
y_train_raw = pd.read_csv(DATA_DIR / 'y_train.csv', index_col='ROW_ID').squeeze()
X_test_raw = pd.read_csv(DATA_DIR / 'X_test.csv', index_col='ROW_ID')

# Binarize target: 1 if return > 0, else 0
y_train = (y_train_raw > 0).astype(int)

print(f'Train shape: {X_train_raw.shape}')
print(f'Test shape: {X_test_raw.shape}')
print(f'Target balance: {y_train.mean():.3f} (fraction positive)')

# Quick data inspection
display(X_train_raw.head())

## 3. Exploratory Data Analysis

Understanding the data before feature engineering

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Target distribution
y_train.value_counts().plot(kind='bar', ax=axes[0,0], title='Target Distribution')
axes[0,0].set_xlabel('Class')
axes[0,0].set_ylabel('Count')

# Return distribution (original)
axes[0,1].hist(y_train_raw, bins=100, edgecolor='none')
axes[0,1].axvline(0, color='red', lw=2)
axes[0,1].set_title('Original Return Distribution')
axes[0,1].set_xlabel('Return')

# Recent returns (RET_1)
axes[0,2].hist(X_train_raw['RET_1'], bins=100, edgecolor='none')
axes[0,2].set_title('Most Recent Return (RET_1)')

# Turnover by group
X_train_raw.groupby('GROUP')['MEDIAN_DAILY_TURNOVER'].mean().plot(
    kind='bar', ax=axes[1,0], title='Avg Turnover by Group'
)

# Correlation heatmap (sample)
corr_cols = ['RET_1', 'RET_5', 'RET_10', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER']
sns.heatmap(X_train_raw[corr_cols].corr(), annot=True, ax=axes[1,1], cmap='coolwarm')
axes[1,1].set_title('Feature Correlations')

# Autocorrelation
rets = X_train_raw[RET_COLS].values[:5000]
autocorrs = [pd.Series(rets[i]).autocorr(1) for i in range(len(rets))]
axes[1,2].hist(autocorrs, bins=50)
axes[1,2].axvline(0, color='red', lw=2)
axes[1,2].set_title('Lag-1 Autocorrelation')

plt.tight_layout()
plt.show()

## 4. Feature Engineering

Creating ~80 features across 7 categories:
1. Momentum & Mean Reversion
2. Volatility & Regime
3. Distribution Shape
4. Signed Volume
5. Return-Volume Interaction
6. Turnover
7. Group Encoding

In [ ]:
# Import feature engineering function from pipeline
import sys
sys.path.append('../utils')
from pipeline import build_features

# Build features
print('Building features for training set...')
X_train_feat = build_features(X_train_raw)

print('Building features for test set...')
X_test_feat = build_features(X_test_raw)

print(f'\nFeatures created: {X_train_feat.shape[1]}')
print(f'Feature names: {list(X_train_feat.columns[:20])}...')

## 5. Train-Validation Split (Temporal)

In [ ]:
from pipeline import temporal_split

# Temporal split (last 15% for validation)
X_tr, y_tr, X_val, y_val = temporal_split(
    X_train_feat, y_train, X_train_raw, val_ratio=0.15
)

print(f'Training set: {X_tr.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Validation target balance: {y_val.mean():.3f}')

## 6. Model Training - LightGBM with Optuna

In [ ]:
from pipeline import tune_lgbm, train_lgbm

# Hyperparameter optimization
print('Running Optuna hyperparameter optimization (50 trials)...')
best_params = tune_lgbm(X_tr, y_tr, X_val, y_val, n_trials=50)

print('\nBest parameters:')
for param, value in best_params.items():
    print(f'  {param}: {value}')

# Train final model
print('\nTraining LightGBM with best parameters...')
lgbm_model = train_lgbm(X_tr, y_tr, X_val, y_val, best_params)

## 7. Model Training - Logistic Regression

In [ ]:
from pipeline import train_logreg

print('Training Logistic Regression...')
logreg_model, scaler = train_logreg(X_tr, y_tr, X_val, y_val)

## 8. Feature Importance (SHAP Analysis)

In [ ]:
from pipeline import shap_analysis
import shap

# SHAP values for LightGBM
print('Computing SHAP values...')
importance_df = shap_analysis(lgbm_model, X_val, n_samples=5000)

# Visualizations
explainer = shap.TreeExplainer(lgbm_model)
sample = X_val.sample(min(1000, len(X_val)), random_state=SEED)
shap_values = explainer.shap_values(sample)

if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class

# Summary plot
shap.summary_plot(shap_values, sample, plot_type='bar', show=False)
plt.tight_layout()
plt.show()

# Bee swarm plot
shap.summary_plot(shap_values, sample, show=False)
plt.tight_layout()
plt.show()

## 9. Ensemble Predictions

In [ ]:
from pipeline import ensemble_predict

# Validation predictions
prob_lgbm_val = lgbm_model.predict_proba(X_val)[:, 1]
prob_logreg_val = logreg_model.predict_proba(scaler.transform(X_val))[:, 1]

# Ensemble (weighted average)
ens_preds_val = ensemble_predict(
    [prob_lgbm_val, prob_logreg_val],
    weights=[0.75, 0.25]
)

# Individual model accuracies
lgbm_acc = accuracy_score(y_val, (prob_lgbm_val > 0.5).astype(int))
logreg_acc = accuracy_score(y_val, (prob_logreg_val > 0.5).astype(int))
ens_acc = accuracy_score(y_val, ens_preds_val)

print(f'LightGBM validation accuracy: {lgbm_acc:.4f}')
print(f'LogReg validation accuracy: {logreg_acc:.4f}')
print(f'Ensemble validation accuracy: {ens_acc:.4f}')

# Comparison plot
pd.Series({'LightGBM': lgbm_acc, 'LogReg': logreg_acc, 'Ensemble': ens_acc}).plot(
    kind='bar', title='Model Comparison (Validation Accuracy)', ylim=(0.5, 0.6)
)
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Final Training and Test Predictions

In [ ]:
# Retrain on full training set for final submission
print('Retraining on full training set...')

# LightGBM
final_lgbm = lgb.LGBMClassifier(
    objective='binary',
    seed=SEED,
    n_estimators=lgbm_model.best_iteration_ + 100,
    **best_params
)
final_lgbm.fit(X_train_feat, y_train)

# LogReg
scaler_full = StandardScaler()
final_logreg = LogisticRegression(C=0.1, max_iter=1000, random_state=SEED, n_jobs=-1)
final_logreg.fit(scaler_full.fit_transform(X_train_feat), y_train)

# Test predictions
prob_lgbm_test = final_lgbm.predict_proba(X_test_feat)[:, 1]
prob_logreg_test = final_logreg.predict_proba(scaler_full.transform(X_test_feat))[:, 1]

test_preds = ensemble_predict(
    [prob_lgbm_test, prob_logreg_test],
    weights=[0.75, 0.25]
)

print(f'Test predictions generated: {len(test_preds)}')
print(f'Fraction of positive predictions: {test_preds.mean():.3f}')

## 11. Generate Submission

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'ROW_ID': X_test_raw.index,
    'TARGET': test_preds
})

# Save
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

print('✓ Submission file created')
print(f'  Location: {OUTPUT_DIR / "submission.csv"}')
print(f'  Rows: {len(submission)}')
print('\nFirst few predictions:')
display(submission.head(10))

## 12. Summary

### Model Performance
- **LightGBM**: Gradient boosting with optimized hyperparameters
- **LogReg**: Linear baseline model
- **Ensemble**: Weighted average (75% LGBM, 25% LogReg)

### Key Features
1. Recent returns (momentum)
2. Volatility measures
3. Mean reversion signals
4. Volume confirmation
5. Group effects

### Next Steps
- Analyze prediction errors
- Test on different market regimes
- Add cross-sectional features
- Experiment with alternative models